# 1. Setup & Imports

In [ ]:
import os
from pathlib import Path

os.environ["STATSBOMB_DATA"] = str(Path(".").resolve() / "data" / "statsbomb" / "data")

import statsbombpy.sb as sb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from features import build_features, get_feature_columns

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Setup complete.")

# 2. Load Data

Load every available competition and pull every shot from the local
StatsBomb mirror.

In [ ]:
competitions = sb.competitions()
print(f"Available competitions: {len(competitions)}")
display(competitions[["competition_name", "season_name", "competition_id", "season_id"]].drop_duplicates().head(20))

In [ ]:
import json
import warnings
from tqdm.auto import tqdm

SHOTS_CACHE = Path("data") / "shots_raw.parquet"
STATSBOMB_ROOT = Path(os.environ["STATSBOMB_DATA"])


def _attach_key_pass_info(shots: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Attach key-pass properties (cross, cut_back, through_ball, high_pass)
    of the pass event to every shot."""
    passes = events[events["type"] == "Pass"].set_index("id")

    def _lookup(kp_id, col, default=False):
        if pd.isna(kp_id) or kp_id not in passes.index:
            return default
        val = passes.at[kp_id, col] if col in passes.columns else default
        return default if pd.isna(val) else val

    kp = shots["shot_key_pass_id"]
    shots["kp_cross"]        = kp.apply(lambda k: bool(_lookup(k, "pass_cross", False)))
    shots["kp_cut_back"]     = kp.apply(lambda k: bool(_lookup(k, "pass_cut_back", False)))
    shots["kp_through_ball"] = kp.apply(lambda k: bool(_lookup(k, "pass_through_ball", False)))
    shots["kp_high_pass"]    = kp.apply(lambda k: _lookup(k, "pass_height", "") == "High Pass")
    return shots


def _collect_local_match_ids() -> list[int]:
    """Read every match_id directly from the local StatsBomb match JSON files."""
    ids: set[int] = set()
    for matches_file in (STATSBOMB_ROOT / "matches").glob("*/*.json"):
        try:
            with open(matches_file) as f:
                for m in json.load(f):
                    ids.add(m["match_id"])
        except (OSError, json.JSONDecodeError):
            continue
    return sorted(ids)


if SHOTS_CACHE.exists():
    shots_raw = pd.read_parquet(SHOTS_CACHE)
    print(f"Cache found: {len(shots_raw)} shots loaded from {SHOTS_CACHE}.")
    if "kp_cross" not in shots_raw.columns:
        print("Warning: cache does not contain kp_* columns. "
              "Delete data/shots_raw.parquet and re-run to enable key-pass features.")
else:
    match_ids = _collect_local_match_ids()
    print(f"Found {len(match_ids)} matches in the local StatsBomb mirror.")

    all_shots: list[pd.DataFrame] = []
    failed = 0

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for match_id in tqdm(match_ids, desc="Loading shots", unit="match"):
            try:
                events = sb.events(match_id=match_id)
                shots = events[events.type == "Shot"].copy()
                if not shots.empty:
                    shots = _attach_key_pass_info(shots, events)
                    all_shots.append(shots)
            except Exception:
                failed += 1
                continue

    shots_raw = pd.concat(all_shots, ignore_index=True)
    print(f"\nLoaded: {len(shots_raw)} shots from {len(match_ids) - failed} matches "
          f"({failed} skipped).")

    SHOTS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shots_raw.to_parquet(SHOTS_CACHE, index=False)
    print(f"Cache written: {SHOTS_CACHE}")

print(f"\nColumns: {list(shots_raw.columns)[:10]}...")
print(f"Shot types: {shots_raw['shot_type'].value_counts().to_dict()}")

# 3. Filter and Clean Shots

- Exclude penalties (their xG is a constant ~0.76)
- Exclude own goals
- Create the binary target column `goal`
- Extract x/y coordinates from the `location` field

In [ ]:
# Drop penalties and own goals
shots = shots_raw[shots_raw["shot_type"] != "Penalty"].copy()
shots = shots[shots["shot_outcome"] != "Own Goal"].copy()

# Target variable
shots["goal"] = (shots["shot_outcome"] == "Goal").astype(int)

# Extract coordinates from location (Parquet returns np.ndarray, not list)
def _coord(loc, i):
    if loc is None:
        return np.nan
    try:
        if len(loc) >= 2:
            return float(loc[i])
    except TypeError:
        return np.nan
    return np.nan

shots["x"] = shots["location"].apply(lambda loc: _coord(loc, 0))
shots["y"] = shots["location"].apply(lambda loc: _coord(loc, 1))

shots = shots.dropna(subset=["x", "y"])

print(f"Shots after cleaning: {len(shots)}")
print(f"Goal rate: {shots['goal'].mean():.1%}")
print(f"\nDistribution of shot_outcome:")
print(shots["shot_outcome"].value_counts())

# 4. Feature Engineering

Apply every feature group via `build_features()` from `features.py`.

In [ ]:
import importlib
import features
importlib.reload(features)
from features import build_features, get_feature_columns

shots = build_features(shots)
feature_cols = get_feature_columns()

print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

print(f"\nNaN share per feature:")
print(shots[feature_cols].isnull().mean().to_string())

# ---- Diagnostic: what is actually inside shot_freeze_frame? ----
print("\n--- freeze_frame diagnostic ---")
sample = shots["shot_freeze_frame"].dropna().head(1)
if len(sample):
    ff = sample.iloc[0]
    print(f"Outer type:   {type(ff).__name__}")
    if hasattr(ff, "__len__") and len(ff) > 0:
        p0 = ff[0]
        print(f"Inner type:   {type(p0).__name__}")
        if hasattr(p0, "keys"):
            print(f"Inner keys:   {list(p0.keys())}")
        loc = p0.get("location") if hasattr(p0, "get") else None
        print(f"location:     {loc!r} ({type(loc).__name__})")
else:
    print("No non-null freeze_frames in the DataFrame!")

# 5. Train / Test Split

Stratified split by goal rate (80 / 20).

In [ ]:
X = shots[feature_cols].copy()
y = shots["goal"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {len(X_train)} shots ({y_train.mean():.1%} goals)")
print(f"Test set:     {len(X_test)} shots ({y_test.mean():.1%} goals)")

# 6. Model: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

rf_proba = rf.predict_proba(X_test)[:, 1]
print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}")
print(f"Random Forest Brier:   {brier_score_loss(y_test, rf_proba):.4f}")

# 7. Model: XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    device="cpu",
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
print(f"\nXGBoost ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"XGBoost Brier:   {brier_score_loss(y_test, xgb_proba):.4f}")

# 8. Comparison: Our Models vs. StatsBomb xG

Compare our two models against StatsBomb's published xG, which is shipped
directly inside every shot event.

In [ ]:
sb_xg = shots.loc[X_test.index, "shot_statsbomb_xg"]

def evaluate_model(name, y_true, y_pred_proba):
    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_true, y_pred_proba),
        "Brier Score": brier_score_loss(y_true, y_pred_proba),
        "Log Loss": log_loss(y_true, y_pred_proba),
    }

results = pd.DataFrame([
    evaluate_model("Random Forest", y_test, rf_proba),
    evaluate_model("XGBoost", y_test, xgb_proba),
    evaluate_model("StatsBomb xG", y_test, sb_xg),
])

display(results.style.format({
    "ROC-AUC": "{:.4f}",
    "Brier Score": "{:.4f}",
    "Log Loss": "{:.4f}",
}).highlight_min(subset=["Brier Score", "Log Loss"], color="lightgreen")
 .highlight_max(subset=["ROC-AUC"], color="lightgreen"))

# 9. Calibration and Residual Analysis

In [ ]:
def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """95% Wilson score interval for a binomial proportion."""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, center - margin), min(1.0, center + margin)


def binned_calibration(y_true, y_pred, n_bins: int = 10) -> pd.DataFrame:
    """Equal-width bins on [0, 1]. Per bin: predicted_mean, observed_rate,
    Wilson CI (lo, hi), shot count."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_pred, edges) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        k = int(y_true[mask].sum())
        lo, hi = wilson_interval(k, n)
        rows.append((y_pred[mask].mean(), k / n, lo, hi, n))
    return pd.DataFrame(rows, columns=["pred_mean", "obs_rate", "lo", "hi", "n"])


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Calibration curve with 95% Wilson CI ---
ax = axes[0]
models = [
    ("Random Forest", rf_proba,     "#1f77b4"),
    ("XGBoost",       xgb_proba,    "#ff7f0e"),
    ("StatsBomb xG",  sb_xg.values, "#2ca02c"),
]
for name, proba, color in models:
    cal = binned_calibration(y_test.values, proba, n_bins=10)
    ax.fill_between(cal["pred_mean"], cal["lo"], cal["hi"],
                    color=color, alpha=0.18, linewidth=0)
    ax.plot(cal["pred_mean"], cal["obs_rate"],
            marker="o", color=color, label=name, lw=2, markersize=7)

ax.plot([0, 1], [0, 1], "k--", label="Perfect", alpha=0.5)
ax.set_xlabel("Predicted xG")
ax.set_ylabel("Actual Goal Rate")
ax.set_title("Calibration Curve  ·  95% Wilson Confidence Bands")
ax.legend(loc="upper left")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# --- Residual analysis ---
ax2 = axes[1]
for name, proba in [("Random Forest", rf_proba), ("XGBoost", xgb_proba)]:
    residuals = y_test.values - proba
    ax2.scatter(proba, residuals, alpha=0.1, s=5, label=name)
ax2.axhline(y=0, color="k", linestyle="--")
ax2.set_xlabel("Predicted xG")
ax2.set_ylabel("Residual (Actual - Predicted)")
ax2.set_title("Residual Plot")
ax2.legend()

plt.tight_layout()
plt.show()

# Bin sizes for diagnosing the mid-range wobble
print("XGBoost  ·  bin counts (equal-width, 10 bins):")
print(binned_calibration(y_test.values, xgb_proba, n_bins=10)
      [["pred_mean", "n", "obs_rate", "lo", "hi"]]
      .round(3).to_string(index=False))

## 9.1 Post-hoc Isotonic Calibration of XGBoost

The calibration curve above typically shows XGBoost slightly off the diagonal
in the mid-to-high xG range. We can fix that without retraining the structure
of the model by fitting a monotonic isotonic regression on top of its raw
predictions.

`CalibratedClassifierCV` does this via internal cross-validation: it fits
XGBoost on 4/5 of the training data, calibrates on the remaining 1/5, and
repeats 5 times. The result is a model whose predicted probabilities are
realigned to the empirical goal rate.

Expected effect: small but consistent **Brier and Log Loss improvements**
(typically 0.001-0.003 reduction). ROC-AUC stays effectively unchanged
because isotonic regression is monotonic — it cannot change ranking, only
the absolute value of each prediction.

Note: this cell trains XGBoost 5 times internally, runtime ~5-10 minutes.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Wrap a fresh XGBoost in isotonic post-hoc calibration.
# cv=5: internally splits X_train into 5 folds; XGB is fit on 4/5,
# the isotonic regressor is fit on the held-out 1/5. The 5 calibrated
# classifiers are then averaged at prediction time.
xgb_calibrator_base = xgb.XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42, device="cpu",
)
xgb_calibrated = CalibratedClassifierCV(
    xgb_calibrator_base, method="isotonic", cv=5,
)
xgb_calibrated.fit(X_train, y_train)
xgb_calib_proba = xgb_calibrated.predict_proba(X_test)[:, 1]

# --- Before / after metrics ---
print("Post-hoc isotonic calibration of XGBoost")
print("=" * 65)
print(f"{'Variant':<40}  {'AUC':>7}  {'Brier':>7}  {'LogLoss':>8}")
print("-" * 65)
for name, proba in [
    ("XGBoost (raw)",                  xgb_proba),
    ("XGBoost (isotonic calibrated)",  xgb_calib_proba),
]:
    print(f"{name:<40}  "
          f"{roc_auc_score(y_test, proba):.4f}  "
          f"{brier_score_loss(y_test, proba):.4f}  "
          f"{log_loss(y_test, proba):.4f}")

# --- Calibration plot before/after ---
fig, ax = plt.subplots(figsize=(9, 7), facecolor="#f7f7f7")
specs = [
    ("XGBoost (raw)",        xgb_proba,        "#ff7f0e"),
    ("XGBoost (isotonic)",   xgb_calib_proba,  "#2ca02c"),
]
for name, proba, color in specs:
    cal = binned_calibration(y_test.values, proba, n_bins=10)
    ax.fill_between(cal["pred_mean"], cal["lo"], cal["hi"],
                    color=color, alpha=0.18, linewidth=0)
    ax.plot(cal["pred_mean"], cal["obs_rate"],
            marker="o", color=color, label=name, lw=2.2, markersize=8)
ax.plot([0, 1], [0, 1], "k--", label="Perfect", alpha=0.5)
ax.set_xlabel("Predicted xG", fontsize=11)
ax.set_ylabel("Actual Goal Rate", fontsize=11)
ax.set_title("Calibration: Before vs. After Isotonic Post-Processing\n"
             "(green should sit tighter on the diagonal than orange)",
             fontsize=13, fontweight="bold", pad=10)
ax.legend(loc="upper left", fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# 10. Feature Importance (SHAP)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, show=False)
axes[0].set_title("SHAP Summary Plot (XGBoost)")

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
axes[1].set_title("SHAP Feature Importance (XGBoost)")

plt.tight_layout()
plt.show()

# 11. PCA — Inspect Feature Redundancy

**Why this section first?** Our feature engineering produced 20 features.
Several of them are redundant **by construction** — for example, the
one-hot `situation_*` columns sum to 1, and `angle_to_goal_rad` is just
`angle_to_goal * pi/180`. Before we can reason about which features
matter, we need to know the *effective dimensionality* of the input.

PCA answers three concrete questions:

- **How many independent directions does our feature space really have?**
  (cumulative explained variance / scree plot)
- **Which features cluster together?**  (loadings heatmap)
- **Are there any components with near-zero variance?**  These are smoking
  guns for exact linear dependencies between features.

The output **motivates Section 12** (a compact 17-component model) and
informs how we set up the coverage-feature ablation in Section 13.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- Standardize (features have very different scales) ---
X_all = shots[feature_cols].copy()
scaler = StandardScaler()
X_std = scaler.fit_transform(X_all)

# --- PCA with all components ---
pca = PCA(n_components=len(feature_cols))
pca.fit(X_std)
explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

loadings = pd.DataFrame(
    pca.components_,
    columns=feature_cols,
    index=[f"PC{i+1}" for i in range(len(feature_cols))],
)

corr = X_all.corr()

# --- Plot ---
fig = plt.figure(figsize=(20, 16), facecolor="#f7f7f7")
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.4], width_ratios=[1, 1.05],
                      hspace=0.32, wspace=0.22)

# (1) Correlation heatmap (lower triangle)
ax1 = fig.add_subplot(gs[0, 0])
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            ax=ax1, cbar_kws={"shrink": 0.7}, square=True,
            xticklabels=True, yticklabels=True)
ax1.set_title("Pearson Correlation Matrix", fontsize=13, fontweight="bold", pad=10)
ax1.tick_params(axis="x", rotation=90, labelsize=8)
ax1.tick_params(axis="y", labelsize=8)

# (2) Scree + cumulative variance
ax2 = fig.add_subplot(gs[0, 1])
x = np.arange(1, len(explained) + 1)
ax2.bar(x, explained, color="#1f77b4", alpha=0.78, label="per PC")
ax2.set_xlabel("Principal Component")
ax2.set_ylabel("Share of explained variance", color="#1f77b4")
ax2.tick_params(axis="y", labelcolor="#1f77b4")
ax2.set_xticks(x)
ax2.tick_params(axis="x", labelsize=8)
ax2b = ax2.twinx()
ax2b.plot(x, cum_explained, color="#d62728", marker="o", lw=2, label="cumulative")
ax2b.axhline(0.95, color="gray", ls="--", lw=1, alpha=0.6)
ax2b.text(len(explained) - 0.5, 0.96, "95%", fontsize=9, color="gray", va="bottom", ha="right")
ax2b.set_ylabel("Cumulative", color="#d62728")
ax2b.tick_params(axis="y", labelcolor="#d62728")
ax2b.set_ylim(0, 1.05)
ax2.set_title("Scree Plot  ·  per-PC + cumulative", fontsize=13, fontweight="bold", pad=10)
ax2.grid(axis="y", alpha=0.3)

# (3) Loadings heatmap (first n PCs)
ax3 = fig.add_subplot(gs[1, :])
n_show = min(12, len(feature_cols))
sns.heatmap(loadings.iloc[:n_show].T, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax3, annot=True, fmt=".2f",
            cbar_kws={"shrink": 0.6}, annot_kws={"size": 8})
ax3.set_title(f"PCA Loadings — first {n_show} principal components "
              "(positive = red, negative = blue)",
              fontsize=13, fontweight="bold", pad=10)
ax3.set_xlabel(""); ax3.set_ylabel("")

plt.tight_layout()
plt.show()

# --- Numeric diagnostics ---
n_95 = int(np.argmax(cum_explained >= 0.95) + 1)
n_99 = int(np.argmax(cum_explained >= 0.99) + 1)
print(f"Effective dimensionality ({len(feature_cols)} features):")
print(f"  95% of variance reached at {n_95} components")
print(f"  99% of variance reached at {n_99} components")

# Strongly correlated feature pairs
print("\nStrongest pairwise linear dependencies (|r| > 0.7):")
pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            pairs.append((feature_cols[i], feature_cols[j], r))
pairs.sort(key=lambda t: -abs(t[2]))
if pairs:
    for f1, f2, r in pairs:
        print(f"  {f1:<36} <-> {f2:<36} r = {r:+.3f}")
else:
    print("  (none)")

# PCs with very low variance = linear dependency between features
near_zero = [(i + 1, e) for i, e in enumerate(explained) if e < 0.005]
if near_zero:
    print(f"\nComponents with <0.5% variance "
          f"(indicator of near-perfect linear dependencies):")
    for pc_i, e in near_zero:
        top = loadings.iloc[pc_i - 1].abs().nlargest(3).index.tolist()
        print(f"  PC{pc_i}: {e * 100:.3f}%   dominant features: {', '.join(top)}")
else:
    print("\nNo PC with <0.5% variance — no hard linear dependencies.")

# 12. PCA-XGBoost: 17 Components

**Why a compact model?** Section 11 showed that our 20 features really
live in an effective space of roughly 13–17 dimensions. A pipeline
`StandardScaler → PCA(17) → XGBoost` captures ~95 % of the total feature
variance with essentially no loss of predictive power. This gives us our
**best compact baseline**.

**Why do this BEFORE testing whether the coverage feature helps?** Two
reasons:

1. **Fair-comparison setup.** In Section 13 we will pit a model *with*
   the coverage information against a model *without* it. To make that
   comparison meaningful, the "with" variant must be a *strong, principled*
   model — not just an XGBoost on raw, partially-redundant inputs.
   PCA-17 strips the redundancy and is the strongest such baseline.

2. **It documents how robust the model is to dimensionality reduction.**
   If 17 PCs match the full feature set, we know the model is not
   relying on noise from low-variance dimensions.

To avoid data leakage, PCA is **fit only on `X_train`** and then
`transform`-applied to `X_test`.

In [ ]:
from sklearn.pipeline import Pipeline

N_COMPONENTS = 17

# Pipeline: Standardize -> PCA -> XGBoost input
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=N_COMPONENTS, random_state=42)),
])

X_train_pca = pca_pipeline.fit_transform(X_train)
X_test_pca  = pca_pipeline.transform(X_test)

var_kept = pca_pipeline.named_steps["pca"].explained_variance_ratio_.sum()
print(f"PCA with {N_COMPONENTS} components captures {var_kept:.1%} of the total variance "
      f"(out of {X_train.shape[1]} original features).")

# XGBoost with the same hyperparameters as Section 7
xgb_pca = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    device="cpu",
)
xgb_pca.fit(
    X_train_pca, y_train,
    eval_set=[(X_test_pca, y_test)],
    verbose=100,
)

xgb_pca_proba = xgb_pca.predict_proba(X_test_pca)[:, 1]
print(f"\nXGBoost (PCA-{N_COMPONENTS}) ROC-AUC: {roc_auc_score(y_test, xgb_pca_proba):.4f}")
print(f"XGBoost (PCA-{N_COMPONENTS}) Brier:   {brier_score_loss(y_test, xgb_pca_proba):.4f}")
print(f"XGBoost (PCA-{N_COMPONENTS}) LogLoss: {log_loss(y_test, xgb_pca_proba):.4f}")

In [ ]:
# --- Metric comparison across all four models ---
results_pca = pd.DataFrame([
    evaluate_model("Random Forest",                  y_test, rf_proba),
    evaluate_model("XGBoost (23 features)",          y_test, xgb_proba),
    evaluate_model(f"XGBoost (PCA-{N_COMPONENTS})",  y_test, xgb_pca_proba),
    evaluate_model("StatsBomb xG",                   y_test, sb_xg),
])

display(results_pca.style.format({
    "ROC-AUC": "{:.4f}",
    "Brier Score": "{:.4f}",
    "Log Loss": "{:.4f}",
}).highlight_min(subset=["Brier Score", "Log Loss"], color="lightgreen")
 .highlight_max(subset=["ROC-AUC"], color="lightgreen"))

# --- Visual comparison: calibration + residuals ---
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Calibration with 95% Wilson CI for all 4 models
ax = axes[0]
model_specs = [
    ("Random Forest",                  rf_proba,       "#1f77b4"),
    ("XGBoost",                        xgb_proba,      "#ff7f0e"),
    (f"XGBoost (PCA-{N_COMPONENTS})",  xgb_pca_proba,  "#9467bd"),
    ("StatsBomb xG",                   sb_xg.values,   "#2ca02c"),
]
for name, proba, color in model_specs:
    cal = binned_calibration(y_test.values, proba, n_bins=10)
    ax.fill_between(cal["pred_mean"], cal["lo"], cal["hi"],
                    color=color, alpha=0.18, linewidth=0)
    ax.plot(cal["pred_mean"], cal["obs_rate"],
            marker="o", color=color, label=name, lw=2, markersize=7)

ax.plot([0, 1], [0, 1], "k--", label="Perfect", alpha=0.5)
ax.set_xlabel("Predicted xG")
ax.set_ylabel("Actual Goal Rate")
ax.set_title("Calibration Curve  ·  95% Wilson CI")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# Residual plot
ax2 = axes[1]
residual_specs = [
    ("Random Forest",                  rf_proba,       "#1f77b4"),
    ("XGBoost",                        xgb_proba,      "#ff7f0e"),
    (f"XGBoost (PCA-{N_COMPONENTS})",  xgb_pca_proba,  "#9467bd"),
]
for name, proba, color in residual_specs:
    residuals = y_test.values - proba
    ax2.scatter(proba, residuals, alpha=0.10, s=5, c=color, label=name)
ax2.axhline(y=0, color="k", linestyle="--", lw=1)
ax2.set_xlabel("Predicted xG")
ax2.set_ylabel("Residual (Actual - Predicted)")
ax2.set_title("Residual Plot")
ax2.legend(loc="upper right", fontsize=9, markerscale=3)

plt.tight_layout()
plt.show()

# 13. Ablation: Does the Coverage Feature Pay for Itself?

**The question:** Section 11 quantified feature redundancy; Section 12
built a compact baseline (PCA-17). Now we ask the central question of
this project: out of all engineered features, does the single retained
360-derived feature — `net_open_goal_pct` — actually contribute
predictive signal? Or could we drop it without losing anything?

**The setup.** Train two XGBoost variants with **identical**
hyperparameters on the same data:

| Variant | Features fed to XGBoost |
|---|---|
| **A. PCA-17 (with coverage)** | PCA(17) on all 20 features, including `net_open_goal_pct` |
| **B. No-Coverage** | 19 raw features (everything **except** `net_open_goal_pct`), no PCA |

Variant B has no PCA because the comparison is about features, not
preprocessing.

**Why 5 seeds × 10 folds = 50 paired evaluations?** A single 80/20 split
gives one noisy answer. With 50 paired (train, test) splits we can:

- compute the **Wilcoxon signed-rank test** on the paired metric
  differences — a non-parametric test that tells us whether the gap is
  systematic or just fold-to-fold noise,
- aggregate per-seed averages to confirm the effect doesn't depend on a
  single lucky split.

Total runtime: ~30–60 minutes (100 XGBoost fits). The cell can be skipped
without breaking the rest of the notebook.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# --- The coverage feature under test ---
# After the streamlining step we have exactly ONE 360-derived feature
# in the model. The ablation removes that single feature.
COVERAGE_FEATURES = ["net_open_goal_pct"]
NO_COVERAGE_FEATURES = [f for f in feature_cols if f not in COVERAGE_FEATURES]
print(f"Coverage features excluded in Variant B: {COVERAGE_FEATURES}")
print(f"Variant A (PCA-17):    {len(feature_cols)} features (incl. coverage)")
print(f"Variant B (No-Cov):    {len(NO_COVERAGE_FEATURES)} features (no coverage)")

# --- CV setup: 5 seeds x 10 folds = 50 evaluations per model type ---
N_SPLITS = 10
SEEDS = [42, 7, 1234, 2024, 99]
TOTAL = len(SEEDS) * N_SPLITS
print(f"\n{len(SEEDS)} seeds x {N_SPLITS} folds = {TOTAL} evaluations per model type")
print("Expected runtime: about 30-60 minutes (100 XGBoost fits)")

X_full     = shots[feature_cols].reset_index(drop=True)
X_full_nc  = shots[NO_COVERAGE_FEATURES].reset_index(drop=True)
y_full     = shots["goal"].reset_index(drop=True)
sb_full    = shots["shot_statsbomb_xg"].reset_index(drop=True)

results = {
    "XGBoost (PCA-17, all features)":     {"auc": [], "brier": [], "logloss": [], "seed": [], "fold": []},
    "XGBoost (no coverage features)":     {"auc": [], "brier": [], "logloss": [], "seed": [], "fold": []},
    "StatsBomb xG":                        {"auc": [], "brier": [], "logloss": [], "seed": [], "fold": []},
}

# random_state is set per seed; not inside the fixed params dict
xgb_params = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", device="cpu",
)

pbar = tqdm(total=TOTAL, desc="Seed x Fold", unit="run")
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_full, y_full)):
        X_tr,    X_te    = X_full.iloc[tr_idx],    X_full.iloc[te_idx]
        X_tr_nc, X_te_nc = X_full_nc.iloc[tr_idx], X_full_nc.iloc[te_idx]
        y_tr,    y_te    = y_full.iloc[tr_idx],    y_full.iloc[te_idx]
        sb_te            = sb_full.iloc[te_idx].values

        # PCA-17 pipeline (Variant A: with coverage)
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=17, random_state=seed)),
        ])
        X_tr_pca = pipe.fit_transform(X_tr)
        X_te_pca = pipe.transform(X_te)
        m_pca = xgb.XGBClassifier(random_state=seed, **xgb_params).fit(
            X_tr_pca, y_tr, verbose=False)
        pred_pca = m_pca.predict_proba(X_te_pca)[:, 1]

        # No-coverage model (Variant B: net_open_goal_pct removed)
        m_nc = xgb.XGBClassifier(random_state=seed, **xgb_params).fit(
            X_tr_nc, y_tr, verbose=False)
        pred_nc = m_nc.predict_proba(X_te_nc)[:, 1]

        valid_sb = ~pd.isna(sb_te)

        for key, pred in [
            ("XGBoost (PCA-17, all features)",   pred_pca),
            ("XGBoost (no coverage features)",   pred_nc),
        ]:
            results[key]["auc"].append(roc_auc_score(y_te, pred))
            results[key]["brier"].append(brier_score_loss(y_te, pred))
            results[key]["logloss"].append(log_loss(y_te, pred))
            results[key]["seed"].append(seed)
            results[key]["fold"].append(fold_idx)

        results["StatsBomb xG"]["auc"].append(roc_auc_score(y_te[valid_sb], sb_te[valid_sb]))
        results["StatsBomb xG"]["brier"].append(brier_score_loss(y_te[valid_sb], sb_te[valid_sb]))
        results["StatsBomb xG"]["logloss"].append(log_loss(y_te[valid_sb], sb_te[valid_sb]))
        results["StatsBomb xG"]["seed"].append(seed)
        results["StatsBomb xG"]["fold"].append(fold_idx)

        pbar.update(1)
pbar.close()

print(f"\nDone: {len(results['XGBoost (PCA-17, all features)']['auc'])} "
      f"evaluations per model type.")


In [ ]:
from scipy.stats import wilcoxon

# --- Summary table ---
summary_rows = []
for model_name, scores in results.items():
    row = {"Model": model_name, "N": len(scores["auc"])}
    for metric in ["auc", "brier", "logloss"]:
        m, s = np.mean(scores[metric]), np.std(scores[metric])
        row[metric.upper()] = f"{m:.4f} +- {s:.4f}"
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
print(f"Mean +- standard deviation across {len(SEEDS)} seeds x {N_SPLITS} folds "
      f"= {len(SEEDS)*N_SPLITS} evaluations:")
display(summary_df)

# --- Box plots per metric (with all 50 points) ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5), facecolor="#f7f7f7")
metrics_info = [
    ("auc",     "ROC-AUC",     "higher is better"),
    ("brier",   "Brier Score", "lower is better"),
    ("logloss", "Log Loss",    "lower is better"),
]
model_names = list(results.keys())
colors = ["#9467bd", "#d62728", "#2ca02c"]
rng = np.random.default_rng(42)

for ax, (metric, title, hint) in zip(axes, metrics_info):
    data = [results[m][metric] for m in model_names]
    bp = ax.boxplot(
        data,
        labels=[m.replace(" (", "\n(") for m in model_names],
        patch_artist=True, widths=0.55, showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "yellow",
                   "markersize": 9, "markeredgecolor": "black"},
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color); patch.set_alpha(0.72)
    for median in bp["medians"]:
        median.set_color("black"); median.set_linewidth(2)

    for i, d in enumerate(data):
        jitter = rng.uniform(-0.10, 0.10, size=len(d))
        ax.scatter(i + 1 + jitter, d, color="black", s=14, alpha=0.45, zorder=10)

    ax.set_title(f"{title}\n({hint})", fontsize=13, fontweight="bold", pad=10)
    ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    ax.tick_params(axis="x", labelsize=9)

plt.suptitle(f"Cross-Validation  ·  {len(SEEDS)} seeds x {N_SPLITS} folds "
             f"= {len(SEEDS)*N_SPLITS} evaluations per model",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# --- Paired difference: PCA-17 minus No-Coverage ---
diff_auc     = (np.array(results["XGBoost (PCA-17, all features)"]["auc"])
              - np.array(results["XGBoost (no coverage features)"]["auc"]))
diff_brier   = (np.array(results["XGBoost (PCA-17, all features)"]["brier"])
              - np.array(results["XGBoost (no coverage features)"]["brier"]))
diff_logloss = (np.array(results["XGBoost (PCA-17, all features)"]["logloss"])
              - np.array(results["XGBoost (no coverage features)"]["logloss"]))

# Per-seed aggregation
seeds_arr = np.array(results["XGBoost (PCA-17, all features)"]["seed"])
per_seed = pd.DataFrame(index=["Delta AUC", "Delta Brier", "Delta LogLoss"])
for seed in SEEDS:
    mask = seeds_arr == seed
    per_seed[f"Seed {seed}"] = [diff_auc[mask].mean(),
                                diff_brier[mask].mean(),
                                diff_logloss[mask].mean()]
per_seed["Overall Mean"] = [diff_auc.mean(), diff_brier.mean(), diff_logloss.mean()]
per_seed["Overall Std"]  = [diff_auc.std(),  diff_brier.std(),  diff_logloss.std()]
print("\nPaired difference (PCA-17 minus No-Coverage), aggregated per seed:")
print("(positive Delta AUC  => PCA-17 is better; negative Delta Brier / Delta LogLoss => PCA-17 is better)")
display(per_seed.round(5))

# === Wilcoxon signed-rank test ============================================
# Non-parametric test: is the median of the paired differences != 0?
# H0: median(Delta metric) = 0 (the two models are equivalent)
# H1 (one-sided): the relevant model is genuinely better
wilcoxon_tests = [
    ("Delta AUC",     diff_auc,     "greater", "PCA-17 has higher AUC"),
    ("Delta Brier",   diff_brier,   "less",    "PCA-17 has lower Brier (better)"),
    ("Delta LogLoss", diff_logloss, "less",    "PCA-17 has lower LogLoss (better)"),
]
wilcoxon_rows = []
test_results = {}
for name, vals, alt, h1_text in wilcoxon_tests:
    stat_2s, p_2s = wilcoxon(vals, alternative="two-sided", zero_method="wilcox")
    _,       p_1s = wilcoxon(vals, alternative=alt,         zero_method="wilcox")
    median = float(np.median(vals))
    test_results[name] = {"W": stat_2s, "p_two": p_2s, "p_one": p_1s,
                          "median": median, "alt": alt}
    wilcoxon_rows.append({
        "Metric":           name,
        "Median Diff":      f"{median:+.5f}",
        "W statistic":      f"{stat_2s:.1f}",
        "p (two-sided)":    f"{p_2s:.3g}",
        "p (one-sided)":    f"{p_1s:.3g}",
        "H1":               h1_text,
        "Significant a=0.05": "yes" if p_2s < 0.05 else "no",
        "Significant a=0.01": "yes" if p_2s < 0.01 else "no",
    })
wilcoxon_df = pd.DataFrame(wilcoxon_rows)
print("\n" + "=" * 75)
print("Wilcoxon Signed-Rank Test — PCA-17 vs No-Coverage")
print("=" * 75)
print(f"Sample size: {len(diff_auc)} paired folds")
print("H0: median paired difference = 0  (no difference between models)")
print("H1 (one-sided): 'PCA-17 is better'")
display(wilcoxon_df)

# === Histograms with Wilcoxon p-value per plot ============================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor="#f7f7f7")
hist_info = [
    ("Delta AUC",     diff_auc,     "positive => PCA-17 better"),
    ("Delta Brier",   diff_brier,   "negative => PCA-17 better"),
    ("Delta LogLoss", diff_logloss, "negative => PCA-17 better"),
]
for ax, (name, vals, hint) in zip(axes, hist_info):
    ax.hist(vals, bins=20, color="#9467bd", alpha=0.75, edgecolor="black")
    ax.axvline(0,           color="black",   lw=2,   ls="--", label="No difference (H0)")
    ax.axvline(vals.mean(), color="#d62728", lw=2.5, label=f"Mean = {vals.mean():+.4f}")
    ax.axvline(np.median(vals), color="#ff7f0e", lw=2.5,
               label=f"Median = {np.median(vals):+.4f}")

    tr = test_results[name]
    sig_color = "#2ca02c" if tr["p_two"] < 0.05 else "#888888"
    sig_text = "significant" if tr["p_two"] < 0.05 else "n.s."
    ax.text(0.03, 0.97,
            f"Wilcoxon Signed-Rank\n"
            f"W = {tr['W']:.0f}\n"
            f"p (two-sided) = {tr['p_two']:.3g}\n"
            f"p (one-sided) = {tr['p_one']:.3g}\n"
            f"-> {sig_text} (a=0.05)",
            transform=ax.transAxes, fontsize=9, va="top", ha="left",
            family="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                      edgecolor=sig_color, linewidth=1.6))

    ax.set_title(f"{name}\n({hint})", fontsize=12, fontweight="bold", pad=8)
    ax.set_xlabel(name)
    ax.set_ylabel(f"Number of folds (of {len(vals)})")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3); ax.set_axisbelow(True)

plt.suptitle(f"Paired differences + Wilcoxon test  ·  "
             f"{len(diff_auc)} folds (PCA-17 minus No-Coverage)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# --- Win rates ---
wins_auc   = int((diff_auc   > 0).sum())
wins_brier = int((diff_brier < 0).sum())
wins_ll    = int((diff_logloss < 0).sum())
print(f"\nIn how many of the {len(diff_auc)} folds does PCA-17 win outright?")
print(f"  AUC:     {wins_auc}/{len(diff_auc)}  ({wins_auc/len(diff_auc):.1%})")
print(f"  Brier:   {wins_brier}/{len(diff_brier)}  ({wins_brier/len(diff_brier):.1%})")
print(f"  LogLoss: {wins_ll}/{len(diff_logloss)}  ({wins_ll/len(diff_logloss):.1%})")

print("\nWilcoxon verdict:")
for name in ["Delta AUC", "Delta Brier", "Delta LogLoss"]:
    tr = test_results[name]
    verdict = "PCA-17 is significantly better" if tr["p_one"] < 0.05 else "no significant difference"
    print(f"  {name}: one-sided p = {tr['p_one']:.3g}  ->  {verdict}")

# 14. Deep-Dive: How Does `net_open_goal_pct` Influence the Prediction?

**The question.** Section 13 tells us *whether* the feature helps overall.
Now we look at *how* it helps, with two specific sub-questions:

1. **Is `net_open_goal_pct` redundant with classical geometry** (distance,
   angle)? If yes, the gain in Section 13 might just be PCA noise. If no,
   the feature is adding genuinely new information.

2. **When does the feature matter most?** Intuition says: when the goal
   mouth is mostly free, the model relies more heavily on the open-angle
   quality, so the coverage feature should carry *more* weight at high
   `net_open_goal_pct` values. Does the SHAP dependence plot confirm
   this?

All plots are based on the SHAP values computed in Section 10.

In [ ]:
focus_features = ["net_open_goal_pct", "distance_to_goal", "angle_to_goal"]
missing = [f for f in focus_features if f not in X_test.columns]
if missing:
    raise ValueError(f"Missing feature columns: {missing} — is shot_lane enabled?")

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test.columns,
    name="mean_|SHAP|",
).sort_values(ascending=False)

print("Mean importance (mean |SHAP|) of the focus features:")
print(mean_abs_shap.loc[focus_features].to_string())
print(f"\nRank within the full feature set:")
for f in focus_features:
    print(f"  {f}: rank {list(mean_abs_shap.index).index(f) + 1} "
          f"of {len(mean_abs_shap)} features")

# Focus SHAP values
focus_idx = [X_test.columns.get_loc(f) for f in focus_features]
shap_focus = pd.DataFrame(shap_values[:, focus_idx], columns=focus_features)
shap_corr = shap_focus.corr()

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5), facecolor="#f7f7f7")

# (1) Importance
ax = axes[0]
mean_abs_shap.loc[focus_features].plot.barh(
    ax=ax, color=["#d62728", "#1f77b4", "#2ca02c"], edgecolor="black")
ax.set_xlabel("Mean absolute SHAP value\n(larger = the feature shifts the "
              "prediction more strongly)", fontsize=10)
ax.set_title("How important is each feature?\n"
             "(comparison: new feature vs. classical geometry)",
             fontsize=12, fontweight="bold", pad=10)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

# (2) Redundancy
ax = axes[1]
sns.heatmap(shap_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, ax=ax, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 13, "weight": "bold"})
ax.set_title("Do the features carry similar information?\n"
             "(+/-1 = features say essentially the same thing; 0 = independent)",
             fontsize=12, fontweight="bold", pad=10)

# (3) Effect as a function of distance
ax = axes[2]
sc = ax.scatter(
    X_test["distance_to_goal"],
    shap_focus["net_open_goal_pct"],
    c=X_test["net_open_goal_pct"],
    cmap="viridis", s=8, alpha=0.5,
)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Distance to goal (yards)", fontsize=10)
ax.set_ylabel("Effect of 'net_open_goal_pct' on the xG prediction\n"
              "(SHAP value: + increases xG, - decreases xG)", fontsize=10)
ax.set_title("When does the feature lift the prediction —\nand when does it lower it?",
             fontsize=12, fontweight="bold", pad=10)
plt.colorbar(sc, ax=ax, label="net_open_goal_pct (open share)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Detail: dependence plot with interaction ---
fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="#f7f7f7")
shap.dependence_plot(
    "net_open_goal_pct", shap_values, X_test,
    interaction_index="distance_to_goal", ax=ax, show=False,
)
ax.set_xlabel("net_open_goal_pct  ·  share of the goal mouth that is "
              "open (0 = fully blocked, 1 = empty net)", fontsize=11)
ax.set_ylabel("SHAP value\n(positive value = raises the xG prediction)", fontsize=11)
ax.set_title("Detail: effect of 'net_open_goal_pct' at different distances\n"
             "(every dot = one shot; color = distance to goal)",
             fontsize=13, fontweight="bold", pad=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 15. Export Models — for the Web App

Save every trained artifact under `models/` so that `app.py` (Streamlit)
can load them:

| File | Content |
|---|---|
| `random_forest.pkl` | Random Forest classifier (pickle) |
| `xgboost.json` | XGBoost trained on all 22 features (raw, uncalibrated) |
| `xgboost_pca.json` | XGBoost trained on the 17 PCA components |
| `pca_pipeline.pkl` | `StandardScaler -> PCA(17)` pipeline (pickle) |
| `xgboost_calibrated.pkl` | XGBoost wrapped in isotonic post-hoc calibration (pickle) |

Only after this section is the web app runnable. Start it with:
```bash
streamlit run app.py
```


In [ ]:
import pickle

os.makedirs("models", exist_ok=True)

# Random Forest
with open("models/random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)

# XGBoost (all features, uncalibrated)
xgb_model.save_model("models/xgboost.json")

# XGBoost-PCA model and pipeline
xgb_pca.save_model("models/xgboost_pca.json")
with open("models/pca_pipeline.pkl", "wb") as f:
    pickle.dump(pca_pipeline, f)

# Isotonic-calibrated XGBoost (from Section 9.1)
with open("models/xgboost_calibrated.pkl", "wb") as f:
    pickle.dump(xgb_calibrated, f)

print("Models exported:")
for p in ["models/random_forest.pkl", "models/xgboost.json",
          "models/xgboost_pca.json", "models/pca_pipeline.pkl",
          "models/xgboost_calibrated.pkl"]:
    size_kb = os.path.getsize(p) / 1024
    print(f"  - {p:<36}  ({size_kb:,.1f} KB)")

print("\nThe web app can now be started:")
print("  streamlit run app.py")
